## 1. Check that the JSON5 parser works

I start with three simple examples: valid JSON5, nested JSON5, and one input with a missing value. This checks that the parser accepts correct input and reports a syntax error for incorrect input before I begin fuzzing.

In [18]:

import json5

test_cases = [
    ("Valid JSON5", '{name: "Alice", age: 25,}'),
    ("Nested JSON5", '{student: {name: "Bob", scores: [80, 90, 95]}}'),
    ("Invalid JSON5", '{name: "Alice", age: }')
]

for name, test_input in test_cases:
    print("\n" + "=" * 50)
    print(name)
    print("Input:", test_input)

    try:
        result = json5.loads(test_input)
        print("Status: ACCEPTED")
        print("Parsed output:", result)

    except Exception as error:
        print("Status: REJECTED")
        print("Error:", type(error).__name__)
        print("Details:", error)



Valid JSON5
Input: {name: "Alice", age: 25,}
Status: ACCEPTED
Parsed output: {'name': 'Alice', 'age': 25}

Nested JSON5
Input: {student: {name: "Bob", scores: [80, 90, 95]}}
Status: ACCEPTED
Parsed output: {'student': {'name': 'Bob', 'scores': [80, 90, 95]}}

Invalid JSON5
Input: {name: "Alice", age: }
Status: REJECTED
Error: ValueError
Details: <string>:1 Unexpected "}" at column 22


## 2. Define the grammar we will use

Here I describe the JSON5 structures our fuzzer can generate, such as objects, arrays, strings, and numbers. This is a small grammar for the experiment, not the complete JSON5 specification. I will keep it unchanged when comparing the normal and heuristic fuzzers.

In [19]:

# Simplified JSON5 grammar for our project

JSON5_GRAMMAR = {
    "<start>": ["<value>"],

    "<value>": [
        "<object>",
        "<array>",
        "<string>",
        "<number>",
        "true",
        "false",
        "null"
    ],

    "<object>": [
        "{}",
        "{<members>}",
        "{<members>,}"
    ],

    "<members>": [
        "<pair>",
        "<pair>, <members>"
    ],

    "<pair>": ["<key>: <value>"],

    "<key>": [
        "<identifier>",
        "<string>"
    ],

    "<identifier>": [
        "name",
        "age",
        "active",
        "student",
        "score"
    ],

    "<array>": [
        "[]",
        "[<elements>]",
        "[<elements>,]"
    ],

    "<elements>": [
        "<value>",
        "<value>, <elements>"
    ],

    "<string>": [
        '"Alice"',
        '"Bob"',
        '"Hello World"'
    ],

    "<number>": [
        "0", "1", "7", "25", "42", "3.14", "-3"
    ]
}

print("JSON5 grammar created successfully!")
print("Number of grammar rules:", len(JSON5_GRAMMAR))


JSON5 grammar created successfully!
Number of grammar rules: 11


## 3. Set up the normal grammar fuzzer

I use `GrammarFuzzer` from *The Fuzzing Book* with our grammar. The random seed makes this run reproducible, and `max_nonterminals=10` controls expansion size; it is **not** a strict depth limit.

In [20]:
from fuzzingbook.GrammarFuzzer import GrammarFuzzer
import random

random.seed(42)

fuzzer = GrammarFuzzer(
    JSON5_GRAMMAR,
    max_nonterminals=10
)

print("Grammar fuzzer initialized successfully!")

Grammar fuzzer initialized successfully!


## 4. Generate the first batch of normal inputs

The normal fuzzer generates **2,000** inputs and stores them in `generated_inputs`. I print every input here to see what the grammar produces. This creates a long notebook output, so later experiments only print a few examples.

In [21]:
generated_inputs = []

for i in range(2000):
    test_input = fuzzer.fuzz()
    generated_inputs.append(test_input)

    print(f"Input {i + 1}: {test_input}")

Input 1: false
Input 2: [true,]
Input 3: []
Input 4: "Alice"
Input 5: {age: 1, "Alice": true, "Bob": {}, "Hello World": [false, true, true], age: "Hello World", student: null}
Input 6: 42
Input 7: true
Input 8: "Hello World"
Input 9: 42
Input 10: false
Input 11: true
Input 12: null
Input 13: []
Input 14: 25
Input 15: 1
Input 16: null
Input 17: [null,]
Input 18: false
Input 19: "Hello World"
Input 20: "Hello World"
Input 21: "Hello World"
Input 22: [null, null,]
Input 23: true
Input 24: [1, null]
Input 25: null
Input 26: {}
Input 27: null
Input 28: []
Input 29: false
Input 30: 1
Input 31: "Bob"
Input 32: false
Input 33: true
Input 34: false
Input 35: null
Input 36: true
Input 37: {age: 0, "Bob": null, age: 25}
Input 38: "Alice"
Input 39: false
Input 40: []
Input 41: 0
Input 42: false
Input 43: {"Alice": {active: true, "Bob": "Hello World"},}
Input 44: false
Input 45: true
Input 46: false
Input 47: false
Input 48: {}
Input 49: {active: false}
Input 50: 25
Input 51: [true, "Hello World", 

## 5. Test the first batch against JSON5

I send the generated inputs to `json5.loads()` and record whether each one is accepted or rejected. The summary counts the outcomes and calculates the acceptance rate. An unexpected exception needs further checking; it does not automatically mean a software bug.

In [22]:

import json5

results = []

for i, test_input in enumerate(generated_inputs, start=1):

    try:
        parsed = json5.loads(test_input)
        status = "ACCEPTED"
        error = None

    except ValueError as e:
        status = "REJECTED"
        error = str(e)

    except Exception as e:
        status = "UNEXPECTED ERROR"
        error = f"{type(e).__name__}: {e}"

    results.append({
        "input": test_input,
        "status": status,
        "error": error
    })

    print(f"Test {i}: {status}")
    print(f"Input: {test_input}")

    if error is not None:
        print(f"Error: {error}")

    print("-" * 40)

# Calculate summary
total = len(results)

accepted = sum(
    r["status"] == "ACCEPTED"
    for r in results
)

rejected = sum(
    r["status"] == "REJECTED"
    for r in results
)

unexpected = sum(
    r["status"] == "UNEXPECTED ERROR"
    for r in results
)

print("\n--- TEST SUMMARY ---")
print("Total inputs:", total)
print("Accepted:", accepted)
print("Rejected:", rejected)
print("Unexpected errors:", unexpected)

print("Acceptance rate:",
      f"{accepted / total * 100:.2f}%")


Test 1: ACCEPTED
Input: false
----------------------------------------
Test 2: ACCEPTED
Input: [true,]
----------------------------------------
Test 3: ACCEPTED
Input: []
----------------------------------------
Test 4: ACCEPTED
Input: "Alice"
----------------------------------------
Test 5: ACCEPTED
Input: {age: 1, "Alice": true, "Bob": {}, "Hello World": [false, true, true], age: "Hello World", student: null}
----------------------------------------
Test 6: ACCEPTED
Input: 42
----------------------------------------
Test 7: ACCEPTED
Input: true
----------------------------------------
Test 8: ACCEPTED
Input: "Hello World"
----------------------------------------
Test 9: ACCEPTED
Input: 42
----------------------------------------
Test 10: ACCEPTED
Input: false
----------------------------------------
Test 11: ACCEPTED
Input: true
----------------------------------------
Test 12: ACCEPTED
Input: null
----------------------------------------
Test 13: ACCEPTED
Input: []
-----------------

## 6. Measure normal grammar production coverage

This time I use `TrackingGrammarCoverageFuzzer` from *The Fuzzing Book*. It generates **1,000** new inputs and records which grammar alternatives were used. The percentage tells me how many different productions appeared at least once, not how much parser code ran.

In [23]:

from fuzzingbook.GrammarCoverageFuzzer import TrackingGrammarCoverageFuzzer
import random

random.seed(42)  # Same results when we repeat the experiment
N = 1000  # Number of inputs to generate

normal_fuzzer = TrackingGrammarCoverageFuzzer(
    JSON5_GRAMMAR,
    max_nonterminals=10  # Controls expansion size
)

normal_inputs = [normal_fuzzer.fuzz() for _ in range(N)]

used = normal_fuzzer.expansion_coverage()  # Rules used
total = normal_fuzzer.max_expansion_coverage()  # All possible rules

print("Generated inputs:", len(normal_inputs))
print("Total grammar productions:", len(total))
print("Used productions:", len(used))
print(f"Grammar coverage: {len(used) / len(total) * 100:.2f}%")
print("Missing productions:", normal_fuzzer.missing_expansion_coverage())


Generated inputs: 1000
Total grammar productions: 36
Used productions: 36
Grammar coverage: 100.00%
Missing productions: set()


## 7. Measure normal parser code coverage

I use `coverage.py` while the JSON5 parser processes the **1,000 `normal_inputs`**. The report shows executed and missed Python statements for the entire `json5` package. It gives a useful baseline percentage, although the total also includes files outside the main parsing path.

In [24]:

import coverage
import json5

cov = coverage.Coverage(source=["json5"], branch=False, data_file=None)  # Track JSON5 code only

cov.start()  # Start measuring coverage

try:
    for test_input in normal_inputs:  # Our 1000 generated inputs
        try:
            json5.loads(test_input)  # Send each input to the parser
        except ValueError:
            pass  # Ignore expected syntax errors
finally:
    cov.stop()  # Stop measuring coverage

print("NORMAL FUZZER - CODE COVERAGE")
normal_code_coverage = cov.report()  # Show executed and missed lines

print(f"\nTotal line coverage: {normal_code_coverage:.2f}%")


NORMAL FUZZER - CODE COVERAGE
Name                                        Stmts   Miss  Cover
---------------------------------------------------------------
.venv\Lib\site-packages\json5\__init__.py       3      3     0%
.venv\Lib\site-packages\json5\__main__.py       4      4     0%
.venv\Lib\site-packages\json5\host.py          29     29     0%
.venv\Lib\site-packages\json5\lib.py          303    256    16%
.venv\Lib\site-packages\json5\parser.py       632    304    52%
.venv\Lib\site-packages\json5\tool.py          61     61     0%
.venv\Lib\site-packages\json5\version.py        2      2     0%
---------------------------------------------------------------
TOTAL                                        1034    659    36%

Total line coverage: 36.27%


## 8. Create the heuristic-guided grammar fuzzer

This class extends the book's `GrammarFuzzer` and changes how it chooses expansions. Objects and arrays get a higher weight than other values, following the weighted-selection idea from Lecture 9. It **encourages** deeper inputs but does not measure the current nesting depth while generating them.

In [25]:

import random
from fuzzingbook.GrammarFuzzer import GrammarFuzzer

class HeuristicGrammarFuzzer(GrammarFuzzer):

    def choose_node_expansion(self, node, children_alternatives):
        symbol, tree = node  # Current grammar symbol
        weights = []  # Weights for available expansions

        for children in children_alternatives:
            expansion = "".join(child[0] for child in children)

            if symbol == "<value>" and expansion in ("<object>", "<array>"):
                weights.append(4)  # Prefer objects and arrays
            else:
                weights.append(1)  # Normal weight for other choices

        return random.choices(
            range(len(children_alternatives)),
            weights=weights
        )[0]

print("Heuristic fuzzer created successfully!")


Heuristic fuzzer created successfully!


## 9. Generate heuristic-guided inputs

I use the same grammar, random seed, input count (**1,000**), and expansion setting as the normal fuzzer. Only the choice of grammar production is different. The first five outputs help me check that generation works.

In [26]:

random.seed(42)  # Same seed as the normal fuzzer
N = 1000  # Generate 1000 test inputs

heuristic_fuzzer = HeuristicGrammarFuzzer(
    JSON5_GRAMMAR,
    max_nonterminals=10  # Same expansion limit as baseline
)

heuristic_inputs = [
    heuristic_fuzzer.fuzz() for _ in range(N)  # Generate inputs
]

print("Total heuristic inputs:", len(heuristic_inputs))
print("\nFirst 5 generated inputs:")

for i, test_input in enumerate(heuristic_inputs[:5], start=1):
    print(f"{i}: {test_input}")


Total heuristic inputs: 1000

First 5 generated inputs:
1: 0
2: []
3: [null, null]
4: true
5: [[{}, {},], {age: [],}]


## 10. Track grammar coverage for the heuristic

I combine the book's grammar-coverage tracker with our heuristic class and generate **1,000** inputs again. This lets me compare production coverage with the normal fuzzer. **Note:** This cell replaces `heuristic_inputs` from the previous cell with the newly tracked batch.

In [27]:

from fuzzingbook.GrammarCoverageFuzzer import TrackingGrammarCoverageFuzzer

class TrackedHeuristicFuzzer(TrackingGrammarCoverageFuzzer, HeuristicGrammarFuzzer):
    pass  # Use both coverage tracking and our heuristic

random.seed(42)  # Repeat with the same seed

tracked_heuristic = TrackedHeuristicFuzzer(
    JSON5_GRAMMAR,
    max_nonterminals=10  # Same limit as the normal fuzzer
)

heuristic_inputs = [
    tracked_heuristic.fuzz() for _ in range(1000)  # Generate and track inputs
]

used = tracked_heuristic.expansion_coverage()  # Productions used
total = tracked_heuristic.max_expansion_coverage()  # All productions

print("HEURISTIC GRAMMAR COVERAGE")
print("Generated inputs:", len(heuristic_inputs))
print("Total productions:", len(total))
print("Used productions:", len(used))
print(f"Grammar coverage: {len(used) / len(total) * 100:.2f}%")
print("Missing productions:", tracked_heuristic.missing_expansion_coverage())


HEURISTIC GRAMMAR COVERAGE
Generated inputs: 1000
Total productions: 36
Used productions: 36
Grammar coverage: 100.00%
Missing productions: set()


## 11. Measure heuristic parser code coverage

Now I run the tracked heuristic inputs through the same JSON5 parser using the same `coverage.py` settings as before. Keeping the measurement method identical makes the normal and heuristic coverage percentages comparable.

In [28]:

import coverage
import json5

heuristic_cov = coverage.Coverage(
    source=["json5"], branch=False, data_file=None  # Track JSON5 lines
)

heuristic_cov.start()  # Start coverage measurement

try:
    for test_input in heuristic_inputs:  # Our 1000 heuristic inputs
        try:
            json5.loads(test_input)  # Test the JSON5 parser
        except ValueError:
            pass  # Expected parsing errors
finally:
    heuristic_cov.stop()  # Stop coverage measurement

print("HEURISTIC FUZZER - CODE COVERAGE")
heuristic_code_coverage = heuristic_cov.report()  # Show coverage report

print(f"\nTotal line coverage: {heuristic_code_coverage:.2f}%")


HEURISTIC FUZZER - CODE COVERAGE
Name                                        Stmts   Miss  Cover
---------------------------------------------------------------
.venv\Lib\site-packages\json5\__init__.py       3      3     0%
.venv\Lib\site-packages\json5\__main__.py       4      4     0%
.venv\Lib\site-packages\json5\host.py          29     29     0%
.venv\Lib\site-packages\json5\lib.py          303    256    16%
.venv\Lib\site-packages\json5\parser.py       632    304    52%
.venv\Lib\site-packages\json5\tool.py          61     61     0%
.venv\Lib\site-packages\json5\version.py        2      2     0%
---------------------------------------------------------------
TOTAL                                        1034    659    36%

Total line coverage: 36.27%


## 12. Compare how complex the inputs are

I parse the normal and heuristic inputs and measure their average nesting depth, maximum depth, and average length. This helps me see whether preferring objects and arrays actually creates more complex test inputs, even when code coverage stays the same.

In [29]:

from statistics import mean
import json5

def nesting_depth(data):
    if isinstance(data, dict):
        return 1 + max((nesting_depth(v) for v in data.values()), default=0)

    if isinstance(data, list):
        return 1 + max((nesting_depth(v) for v in data), default=0)

    return 0  # Simple values have no nesting


def measure_complexity(inputs):
    depths = []
    lengths = []

    for text in inputs:
        data = json5.loads(text)  # Parse the generated input
        depths.append(nesting_depth(data))  # Measure nesting
        lengths.append(len(text))  # Count characters

    return mean(depths), max(depths), mean(lengths)


normal = measure_complexity(normal_inputs)
heuristic = measure_complexity(heuristic_inputs)

print(f"{'Metric':<22} {'Normal':>12} {'Heuristic':>12}")
print("-" * 48)

for i, metric in enumerate(["Average depth", "Maximum depth", "Average length"]):
    print(f"{metric:<22} {normal[i]:>12.2f} {heuristic[i]:>12.2f}")


Metric                       Normal    Heuristic
------------------------------------------------
Average depth                  0.40         1.42
Maximum depth                  8.00        19.00
Average length                 9.85        26.36


## 13. Generate mutated test inputs

This follows the *Grammars as mutation seeds* idea in Lecture 6. I select up to **20** longer inputs from the normal grammar run and let `MutationFuzzer` from *The Fuzzing Book* change their characters. I skip its initial unchanged seeds and collect **1,000** mutated inputs, some of which may be invalid JSON5.

In [30]:

import random
from fuzzingbook.MutationFuzzer import MutationFuzzer

seed_inputs = [s for s in normal_inputs if len(s) > 10][:20]  # Use 20 longer grammar inputs

random.seed(42)  # Reproduce the experiment

mutation_fuzzer = MutationFuzzer(
    seed=seed_inputs,      # Original inputs to mutate
    min_mutations=1,       # Minimum changes per input
    max_mutations=3        # Maximum changes per input
)

for _ in seed_inputs:
    mutation_fuzzer.fuzz()  # Skip unchanged seeds

mutation_inputs = [
    mutation_fuzzer.fuzz() for _ in range(1000)  # Generate 1000 mutated inputs
]

print("Mutation inputs:", len(mutation_inputs))
print("\nFirst 5 mutated inputs:")

for i, text in enumerate(mutation_inputs[:5], 1):
    print(f"{i}: {repr(text)}")


Mutation inputs: 1000

First 5 mutated inputs:
1: '"Helno World"'
2: '{"AlicE": {active: true, "Bob": "Hello World"},}'
3: '"Hello!World"'
4: '{age: 1, "Alice": true, "Bob" {}, "Hello World": [false, true, true], age: "Hello World", student: null}'
5: '{name: {name: {"Bob": {}, student: false, "Alice": "Alice"}}, age: [0, rue, true], score:0nll}'


## 14. Measure mutation coverage and acceptance

I test the mutated strings with JSON5 and measure package line coverage the same way as before. I also count accepted inputs, expected syntax rejections, and other exceptions. Invalid inputs can reach error-handling code that valid grammar inputs do not exercise.

In [31]:

import coverage
import json5

mutation_cov = coverage.Coverage(
    source=["json5"], branch=False, data_file=None  # Track JSON5 code
)

accepted = 0
rejected = 0
unexpected = 0

mutation_cov.start()  # Start measuring coverage

try:
    for test_input in mutation_inputs:  # Our 1000 mutated inputs
        try:
            json5.loads(test_input)  # Run the parser
            accepted += 1

        except ValueError:
            rejected += 1  # Expected parsing error

        except Exception as error:
            unexpected += 1  # Other errors need investigation
            print(type(error).__name__, error)

finally:
    mutation_cov.stop()  # Stop measuring coverage

print("MUTATION FUZZER - CODE COVERAGE")
mutation_code_coverage = mutation_cov.report()

print(f"\nTotal line coverage: {mutation_code_coverage:.2f}%")
print("Accepted:", accepted)
print("Rejected:", rejected)
print("Unexpected errors:", unexpected)
print(f"Acceptance rate: {accepted / len(mutation_inputs) * 100:.2f}%")


MUTATION FUZZER - CODE COVERAGE
Name                                        Stmts   Miss  Cover
---------------------------------------------------------------
.venv\Lib\site-packages\json5\__init__.py       3      3     0%
.venv\Lib\site-packages\json5\__main__.py       4      4     0%
.venv\Lib\site-packages\json5\host.py          29     29     0%
.venv\Lib\site-packages\json5\lib.py          303    255    16%
.venv\Lib\site-packages\json5\parser.py       632    267    58%
.venv\Lib\site-packages\json5\tool.py          61     61     0%
.venv\Lib\site-packages\json5\version.py        2      2     0%
---------------------------------------------------------------
TOTAL                                        1034    621    40%

Total line coverage: 39.94%
Accepted: 400
Rejected: 600
Unexpected errors: 0
Acceptance rate: 40.00%


## 15. Time how long parsing takes

I time the JSON5 parser on each already-generated input set. Each run is repeated three times, and I report the median to reduce small timing fluctuations. This measures **parsing time only**, without input generation or coverage tracking.

In [32]:

import time
from statistics import median

def measure_time(inputs, repeats=3):
    times = []

    for _ in range(repeats):  # Repeat to reduce timing variation
        start = time.perf_counter()  # Start timer

        for text in inputs:
            try:
                json5.loads(text)  # Run the parser
            except ValueError:
                pass  # Expected invalid inputs

        times.append(time.perf_counter() - start)

    return median(times)  # Use the middle timing result


normal_time = measure_time(normal_inputs)
heuristic_time = measure_time(heuristic_inputs)
mutation_time = measure_time(mutation_inputs)

print("PARSER EXECUTION TIME")
print(f"Normal:    {normal_time:.4f} seconds")
print(f"Heuristic: {heuristic_time:.4f} seconds")
print(f"Mutation:  {mutation_time:.4f} seconds")


PARSER EXECUTION TIME
Normal:    0.2378 seconds
Heuristic: 0.8941 seconds
Mutation:  0.6790 seconds


## 16. Time how long generation takes

I create fresh normal, heuristic, and mutation fuzzers and measure how long each takes to produce **1,000** inputs. These are preliminary timing measurements. The mutation timer does **not** include the time needed to generate its original grammar-based seeds.

In [33]:

import time
import random
from fuzzingbook.GrammarFuzzer import GrammarFuzzer
from fuzzingbook.MutationFuzzer import MutationFuzzer

N = 1000  # Number of inputs for each method

def generation_time(fuzzer, n):
    start = time.perf_counter()  # Start timer
    for _ in range(n):
        fuzzer.fuzz()  # Generate one input
    return time.perf_counter() - start  # Elapsed time

random.seed(42)
normal_gen = GrammarFuzzer(JSON5_GRAMMAR, max_nonterminals=10)
normal_gen_time = generation_time(normal_gen, N)

random.seed(42)
heuristic_gen = HeuristicGrammarFuzzer(JSON5_GRAMMAR, max_nonterminals=10)
heuristic_gen_time = generation_time(heuristic_gen, N)

seed_inputs = [s for s in normal_inputs if len(s) > 10][:20]
mutation_gen = MutationFuzzer(seed=seed_inputs, min_mutations=1, max_mutations=3)

for _ in seed_inputs:
    mutation_gen.fuzz()  # Skip the original seeds

random.seed(42)
mutation_gen_time = generation_time(mutation_gen, N)

print("INPUT GENERATION TIME")
print(f"Normal:    {normal_gen_time:.4f} seconds")
print(f"Heuristic: {heuristic_gen_time:.4f} seconds")
print(f"Mutation:  {mutation_gen_time:.4f} seconds")


INPUT GENERATION TIME
Normal:    0.4282 seconds
Heuristic: 4.6185 seconds
Mutation:  0.0051 seconds


## 17. Summarize time and code coverage



In [34]:

# Combine generation and parsing times
normal_total = normal_gen_time + normal_time
heuristic_total = heuristic_gen_time + heuristic_time
mutation_total = mutation_gen_time + mutation_time

print("TOTAL TESTING TIME (1000 INPUTS)")
print("-" * 40)

print(f"Normal:    {normal_total:.4f} seconds")
print(f"Heuristic: {heuristic_total:.4f} seconds")
print(f"Mutation:  {mutation_total:.4f} seconds")

print("\nCODE COVERAGE")
print(f"Normal:    {normal_code_coverage:.2f}%")
print(f"Heuristic: {heuristic_code_coverage:.2f}%")
print(f"Mutation:  {mutation_code_coverage:.2f}%")


TOTAL TESTING TIME (1000 INPUTS)
----------------------------------------
Normal:    0.6659 seconds
Heuristic: 5.5126 seconds
Mutation:  0.6841 seconds

CODE COVERAGE
Normal:    36.27%
Heuristic: 36.27%
Mutation:  39.94%
